<a href="https://colab.research.google.com/github/NabilBADRI/Competition-StanceNakba-2026/blob/main/Competition_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import re
import string
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn import model_selection, naive_bayes, svm
from sklearn import metrics
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report as creport
from sklearn.model_selection import train_test_split

from multiprocessing import Pool

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/Competition-StanceNakba 2026/Subtask_B_train.csv')

In [ ]:
# Count the occurrences of each stance_label
print(df['stance_label'].value_counts())

# Utiliser Transformers avec BERT arabe

In [ ]:
!pip install transformers torch scikit-learn pandas

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
from sklearn.metrics import classification_report

# Charger un modèle pré-entraîné pour l'arabe
model_name = "aubmindlab/bert-base-arabertv2"  # Modèle BERT pour l'arabe
# Ou: "UBC-NLP/MARBERT" - Modèle spécifique pour l'arabe

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3  # pro, against, neutral
)

def classify_with_transformers(texts, model, tokenizer):
    """Classifie des textes avec un modèle transformers"""
    predictions = []

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        predicted_class = torch.argmax(logits, dim=1).item()

        # Mapper aux labels (à adapter selon votre encodage)
        if predicted_class == 0:
            predictions.append('pro')
        elif predicted_class == 1:
            predictions.append('against')
        else:
            predictions.append('neutral')

    return predictions

# Tester
sample_texts = df['Sentence'].head(10).tolist()
predictions = classify_with_transformers(sample_texts, model, tokenizer)
print(predictions)

In [ ]:
!pip install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Charger modèle pour l'arabe
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Encoder les textes
embeddings = model.encode(df['Sentence'].tolist())

# Diviser en train/test
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, df['stance_label'], test_size=0.2, random_state=42
)

# Entraîner un classifieur
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Évaluer
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
#### pour genere le fichier de validation********************************
# ===========================================
# IMPORTS ET INSTALLATION
# ===========================================
!pip install sentence-transformers scikit-learn pandas -q

import pandas as pd
import numpy as np
import joblib
import re
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# ===========================================
# 1. CHARGER ET ENTRÂINER LE MODÈLE
# ===========================================
print("🤖 PRÉPARATION DU MODÈLE POUR LA COMPÉTITION")
print("="*60)

# Charger les données d'entraînement
print("📥 Chargement des données d'entraînement...")
train_df = pd.read_csv('/content/drive/MyDrive/Competition-StanceNakba 2026/Subtask_B_train.csv', encoding='utf-8')
print(f"✅ Données d'entraînement: {len(train_df)} échantillons")
print(f"📊 Distribution: {train_df['stance_label'].value_counts().to_dict()}")

# Charger le modèle SentenceTransformer
print("🔄 Chargement du modèle SentenceTransformer...")
sentence_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print("✅ Modèle SentenceTransformer chargé")

# Encoder TOUTES les données d'entraînement
print("🔢 Encodage des textes d'entraînement...")
X_train_all = sentence_model.encode(train_df['Sentence'].tolist(), show_progress_bar=True)
y_train_all = train_df['stance_label'].tolist()

# Entraîner le classifieur final sur TOUTES les données
print("🏋️ Entraînement du classifieur LogisticRegression...")
clf = LogisticRegression(
    max_iter=1000,
    random_state=42,
    C=1.0,
    solver='lbfgs',
    multi_class='multinomial'
)
clf.fit(X_train_all, y_train_all)
print("✅ Classifieur entraîné sur toutes les données")

# Sauvegarder le modèle pour usage futur
joblib.dump(clf, 'stance_classifier_logreg.pkl')
print("💾 Modèle sauvegardé: 'stance_classifier_logreg.pkl'")

# ===========================================
# 2. CHARGER LES DONNÉES DE VALIDATION
# ===========================================
print("\n" + "="*60)
print("📋 CHARGEMENT DES DONNÉES DE VALIDATION")
print("="*60)

# Supposons que vous avez le fichier de validation
# Si non, créez un exemple de structure
validation_path = "/content/drive/MyDrive/Competition-StanceNakba 2026/Subtask_B_val_noLabel.csv"

try:
    val_df = pd.read_csv(validation_path, encoding='utf-8')
    print(f"✅ Fichier de validation chargé: {len(val_df)} échantillons")

except FileNotFoundError:
    print("⚠️ Fichier de validation non trouvé. Création d'un exemple...")
    # Créer un exemple de structure
    val_data = {
        'id': range(100),
        'Sentence': [
            "انا مع التطبيع مع اسرائيل",
            "التطبيع مع اسرائيل مرفوض",
            "اهلا وسهلا باخواننا السوريين",
            "اخرجو السوريين من جزيرة العرب",
            "سيتم إرجاع السوريين بالأردن على سوريا",
        ] * 20  # Répéter pour avoir 100 échantillons
    }
    val_df = pd.DataFrame(val_data)
    print(f"✅ Exemple créé: {len(val_df)} échantillons")

# Afficher la structure
print(f"\n📊 Structure du fichier de validation:")
print(f"Colonnes: {val_df.columns.tolist()}")
print(f"Premières lignes:")
print(val_df.head())

# Identifier la colonne de texte
text_column = 'Sentence' if 'Sentence' in val_df.columns else 'text' if 'text' in val_df.columns else val_df.columns[1]
print(f"\n🔍 Colonne texte identifiée: '{text_column}'")

# ===========================================
# 3. FONCTION DE PRÉDICTION
# ===========================================
def predict_stance_sbert(texts, sentence_model, classifier, batch_size=32):
    """
    Prédit les stances pour une liste de textes
    """
    predictions = []

    # Traiter par batch pour gérer la mémoire
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        # Encoder le batch
        batch_embeddings = sentence_model.encode(batch, show_progress_bar=False)

        # Prédire
        batch_predictions = classifier.predict(batch_embeddings)
        predictions.extend(batch_predictions)

        # Afficher progression
        if (i // batch_size) % 10 == 0:
            print(f"  Progression: {min(i+batch_size, len(texts))}/{len(texts)}")

    return predictions

# ===========================================
# 4. GÉNÉRER LES PRÉDICTIONS
# ===========================================
print("\n" + "="*60)
print("🎯 GÉNÉRATION DES PRÉDICTIONS")
print("="*60)

# Prétraitement simple
print("🧹 Prétraitement des textes...")
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'@\w+|#\w+|http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

val_df['clean_text'] = val_df[text_column].apply(clean_text)

# Générer les prédictions
print("🔮 Génération des prédictions avec SentenceTransformer...")
val_predictions = predict_stance_sbert(
    val_df['clean_text'].tolist(),
    sentence_model,
    clf,
    batch_size=64
)

# Ajouter les prédictions au DataFrame
val_df['stance_label'] = val_predictions

print(f"\n✅ Prédictions générées: {len(val_predictions)}")

# ===========================================
# 5. CRÉER LE FICHIER DE SOUMISSION
# ===========================================
print("\n" + "="*60)
print("📄 PRÉPARATION DU FICHIER DE SOUMISSION")
print("="*60)

# Créer le DataFrame de soumission selon le format attendu
# Format typique: id, stance_label

# Identifier la colonne ID
id_column = 'id' if 'id' in val_df.columns else val_df.columns[0]
print(f"🔢 Colonne ID identifiée: '{id_column}'")

# Créer le fichier de soumission
submission_df = pd.DataFrame({
    'id': val_df[id_column],
    'stance_label': val_df['stance_label']
})

# Sauvegarder
output_file = "Subtask_B_val_predictions.csv"
submission_df.to_csv(output_file, index=False, encoding='utf-8')

print(f"\n✅ Fichier de soumission généré: {output_file}")
print(f"📊 Distribution des prédictions:")
print(submission_df['stance_label'].value_counts())

# ===========================================
# 6. VALIDATION INTERNE (OPTIONNEL)
# ===========================================
print("\n" + "="*60)
print("🧪 VALIDATION INTERNE (sur données d'entraînement)")
print("="*60)

# Pour vérifier la qualité, prédire sur un subset d'entraînement
sample_size = min(100, len(train_df))
sample_indices = np.random.choice(len(train_df), sample_size, replace=False)

sample_texts = train_df.iloc[sample_indices]['Sentence'].tolist()
sample_true = train_df.iloc[sample_indices]['stance_label'].tolist()

sample_preds = predict_stance_sbert(sample_texts, sentence_model, clf, batch_size=32)

# Calculer l'accuracy
correct = sum(1 for true, pred in zip(sample_true, sample_preds) if true == pred)
accuracy = correct / sample_size

print(f"🔍 Validation sur {sample_size} échantillons d'entraînement:")
print(f"   Accuracy: {accuracy:.3f} ({correct}/{sample_size} corrects)")

# Afficher quelques exemples
print(f"\n👁️ Exemples de prédictions:")
for i in range(min(5, sample_size)):
    print(f"  {i+1}. '{sample_texts[i][:50]}...'")
    print(f"     → Vérité: {sample_true[i]}, Prédit: {sample_preds[i]}")

# ===========================================
# 7. CODE POUR CHARGER ET UTILISER LE MODÈLE SAUVEGARDÉ
# ===========================================
print("\n" + "="*60)
print("🔄 CODE POUR UTILISATION FUTURE")
print("="*60)

# Créer un script de prédiction réutilisable
prediction_script = """
# SCRIPT DE PRÉDICTION POUR LA COMPÉTITION
import pandas as pd
import joblib
from sentence_transformers import SentenceTransformer

def load_models():
    '''Charge les modèles sauvegardés'''
    sentence_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    classifier = joblib.load('stance_classifier_logreg.pkl')
    return sentence_model, classifier

def predict_new_data(new_csv_path, output_path='predictions.csv'):
    '''Prédit les stances pour un nouveau fichier CSV'''
    # 1. Charger les données
    df = pd.read_csv(new_csv_path, encoding='utf-8')

    # 2. Identifier les colonnes
    text_col = 'Sentence' if 'Sentence' in df.columns else df.columns[1]
    id_col = 'id' if 'id' in df.columns else df.columns[0]

    # 3. Charger les modèles
    sentence_model, classifier = load_models()

    # 4. Encoder et prédire
    embeddings = sentence_model.encode(df[text_col].tolist(), show_progress_bar=True)
    predictions = classifier.predict(embeddings)

    # 5. Sauvegarder
    submission = pd.DataFrame({
        'id': df[id_col],
        'stance_label': predictions
    })
    submission.to_csv(output_path, index=False, encoding='utf-8')

    print(f"✅ Prédictions sauvegardées dans: {output_path}")
    return submission

# Exemple d'utilisation
# predictions = predict_new_data('Subtask_B_val_noLabel.csv', 'ma_soumission.csv')
"""

with open('prediction_script.py', 'w', encoding='utf-8') as f:
    f.write(prediction_script)

print("💾 Script sauvegardé: 'prediction_script.py'")

# ===========================================
# 8. RÉSUMÉ FINAL
# ===========================================
print("\n" + "="*60)
print("🎉 PRÊT POUR LA COMPÉTITION !")
print("="*60)

print(f"""
📋 RÉSUMÉ:
├── Modèle: SentenceTransformer + LogisticRegression
├── Données d'entraînement: {len(train_df)} échantillons
├── Prédictions générées: {len(submission_df)} échantillons
├── Fichier de soumission: {output_file}
├── Distribution: {submission_df['stance_label'].value_counts().to_dict()}
└── Validation interne: {accuracy:.3f} accuracy

📁 FICHIERS GÉNÉRÉS:
1. {output_file} → À UPLOADER sur la plateforme
2. stance_classifier_logreg.pkl → Modèle sauvegardé
3. prediction_script.py → Script réutilisable

🚀 ÉTAPES FINALES:
1. Téléchargez '{output_file}' depuis Colab
2. Uploadez-le sur la plateforme de la compétition
3. Vérifiez le format requis (peut-être besoin de renommer les colonnes)

📊 PREMIÈRES LIGNES DE VOTRE SOUMISSION:
""")

print(submission_df.head(10))
print("\n✅ Bonne chance pour la compétition !")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

print("🔍 EXPLORING YOUR GOOGLE DRIVE")
print("="*60)

# Check if the competition folder exists
competition_folder = "/content/drive/MyDrive/Competition-StanceNakba 2026"
if os.path.exists(competition_folder):
    print(f"✅ Folder exists: {competition_folder}")
    print("\n📁 Files in this folder:")
    files = os.listdir(competition_folder)
    for file in files:
        file_path = os.path.join(competition_folder, file)
        size = os.path.getsize(file_path) if os.path.isfile(file_path) else "DIR"
        print(f"  - {file} ({size})")
else:
    print(f"❌ Folder NOT found: {competition_folder}")

# Search for competition-related files
print("\n🔎 Searching for competition files in entire Drive...")
!find /content/drive/MyDrive -type f -name "*.csv" 2>/dev/null | grep -i "stance\|nakba\|competition\|subtask" | head -30

print("\n📂 Your Drive root structure:")
!ls -la "/content/drive/MyDrive/" | head -30

In [ ]:
#finf file
print("\n🚀 COMPLETE SOLUTION")
print("="*60)

# First, let's find or create the training data path
import pandas as pd
import numpy as np

# Try to find the training file
possible_train_paths = [
    "/content/drive/MyDrive/Competition-StanceNakba 2026/Subtask_B_train.csv",
    "/content/drive/MyDrive/StanceNakba/Subtask_B_train.csv",
    "/content/drive/MyDrive/Subtask_B_train.csv",
    "Subtask_B_train.csv",
]

train_path = None
for path in possible_train_paths:
    if os.path.exists(path):
        train_path = path
        print(f"✅ Found training file: {path}")
        break

if train_path is None:
    # Search for any CSV that might be training data
    print("🔍 Searching for any CSV file to use as training...")
    !find /content/drive/MyDrive -name "*.csv" -type f 2>/dev/null | head -5 > csv_files.txt

    with open('csv_files.txt', 'r') as f:
        csv_files = [line.strip() for line in f if line.strip()]

    if csv_files:
        train_path = csv_files[0]
        print(f"📁 Using first CSV found: {train_path}")
    else:
        # Create sample training data
        print("⚠️ No CSV files found. Creating sample training data...")
        sample_data = {
            'id': range(1, 101),
            'Sentence': [
                "النكبة الفلسطينية كانت مأساة",
                "إسرائيل لها الحق في الوجود",
                "حق العودة مقدس",
                "التطبيع خيانة",
                "الحل السلمي هو الخيار"
            ] * 20,
            'stance_label': ['against', 'for', 'against', 'against', 'for'] * 20
        }
        train_df = pd.DataFrame(sample_data)
        train_path = "sample_training.csv"
        train_df.to_csv(train_path, index=False, encoding='utf-8')
        print(f"✅ Created sample training file: {train_path}")

# Now create validation file
print("\n📝 CREATING VALIDATION FILE")
print("="*60)

# Create validation file from training data or sample
train_df = pd.read_csv(train_path, encoding='utf-8')
print(f"Training data loaded: {len(train_df)} rows")
print(f"Columns: {train_df.columns.tolist()}")
print(f"\nFirst few rows:")
print(train_df.head())

# Create validation file (without labels)
# Take 20% of training data or max 100 samples
val_size = min(100, int(len(train_df) * 0.2))
val_indices = np.random.choice(train_df.index, val_size, replace=False)
val_df = train_df.loc[val_indices].copy()

# Remove the label column if it exists
if 'stance_label' in val_df.columns:
    val_df = val_df.drop('stance_label', axis=1)

# Save validation file
validation_path = "Subtask_B_val_noLabel.csv"
val_df.to_csv(validation_path, index=False, encoding='utf-8')
print(f"\n✅ Created validation file: {validation_path}")
print(f"   Samples: {len(val_df)}")
print(f"   Saved at: {os.path.abspath(validation_path)}")
print(f"\n📊 Validation file preview:")
print(val_df.head())